# AI Narratives Pipeline

本 Notebook 按顺序展示完整的数据处理流程：
1. 数据清洗（Build Transcripts + Clean Text）
2. AI 句子提取
3. Narrative 指标构造
4. 合并财务与收益数据
5. 导出交付文件

**注意：** 由于原始 conference call 数据量大（15 GB），建议使用 `src/` 下的脚本分步运行。本 notebook 用于展示流程逻辑和检查结果。

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE = Path('.').resolve()
if BASE.name != 'AI Narratives':
    BASE = BASE.parent  # adjust if running from notebooks/
print(f'Project root: {BASE}')
print(f'Files: {os.listdir(BASE)}')

## Step 1a: Build Transcripts

将 component-level 的 conference call 按 transcriptid 聚合为 transcript-level，再通过 link table 匹配 gvkey。

脚本：`src/01_build_transcripts.py`

In [ ]:
# 检查 Step 1a 输出
transcripts = pd.read_csv(BASE / 'data_processed' / 'transcripts_firm_quarter.csv', 
                           nrows=5, dtype={'gvkey': str})
print(f'Columns: {list(transcripts.columns)}')
transcripts.head()

In [ ]:
# 完整数据统计
import csv
with open(BASE / 'data_processed' / 'transcripts_firm_quarter.csv') as f:
    reader = csv.reader(f)
    header = next(reader)
    n = sum(1 for _ in reader)
print(f'Total firm-quarter observations: {n:,}')

## Step 1b: Clean Text

文本预处理：lowercase、去标点/数字/停用词、lemmatize、句子分割。

脚本：`src/02_clean_text.py`

In [ ]:
# 检查 Step 1b 输出
cleaned = pd.read_csv(BASE / 'data_processed' / 'transcripts_cleaned.csv',
                       nrows=5, dtype={'gvkey': str})
print(f'Columns: {list(cleaned.columns)}')
print(f'Sample clean text (first 200 chars): {cleaned["text_clean"].iloc[0][:200]}')
cleaned[['gvkey', 'year', 'quarter', 'total_word_count', 'total_sentence_count', 'clean_word_count']].head()

## Step 2: AI Sentence Extraction

基于 AI 词典（32 核心词 + 16 扩展词），按句子级别匹配，提取 AI 相关句子。

脚本：`src/03_extract_ai_sentences.py`

In [ ]:
# 检查 AI 句子
ai_sent = pd.read_csv(BASE / 'data_processed' / 'ai_sentences_raw.csv',
                        dtype={'gvkey': str})
print(f'Total AI sentences: {len(ai_sent):,}')
print(f'Unique firms with AI mentions: {ai_sent["gvkey"].nunique():,}')
print(f'\nSample AI sentences:')
for _, row in ai_sent.head(5).iterrows():
    print(f'  [{row["gvkey"]}] {row["sentence"][:120]}...')

In [ ]:
# 检查 mention 指标
mention = pd.read_csv(BASE / 'data_processed' / 'transcripts_with_ai_mention.csv',
                       nrows=1000, dtype={'gvkey': str})
print(f'% firms with at least 1 AI sentence: {(mention["ai_sentence_count"]>0).mean()*100:.1f}%')
print(f'Mean ai_word_count: {mention["ai_word_count"].mean():.2f}')
print(f'Mean ai_sentence_ratio: {mention["ai_sentence_ratio"].mean():.4f}')

## Step 3: Narrative Measure Construction

三种方法并行构造深度指标：
- Method 1: 词典分类（actionable / speculative / risk / hype / irrelevant）
- Method 2: 语义相似度（sentence-transformers cosine similarity）
- Method 3: LLM 分类或规则降级（adoption / innovation / risk / hype）

脚本：`src/04_build_narrative_measures.py`

In [ ]:
# 检查 narrative 指标
narr = pd.read_csv(BASE / 'data_processed' / 'transcripts_with_narratives.csv',
                     nrows=1000, dtype={'gvkey': str})
narr_cols = [c for c in narr.columns if c.startswith('ai_')]
print(f'AI narrative columns: {narr_cols}')
narr[narr_cols].describe().round(4)

In [ ]:
# 检查句子级标注
sent_level = pd.read_csv(BASE / 'outputs' / 'tables' / 'ai_sentence_level.csv',
                          nrows=1000, dtype={'gvkey': str})
print(f'Label distribution:')
print(sent_level['label'].value_counts())
print(f'\nSemantic score stats:')
print(sent_level['semantic_score'].describe())

## Step 4: Merge Financial & Return Data

合并 Compustat 财务变量和 CRSP 收益率，构造控制变量和 future return。

脚本：`src/05_merge_financials_and_returns.py`

In [ ]:
# 检查合并后的面板
panel = pd.read_csv(BASE / 'data_processed' / 'panel_with_financials.csv',
                      nrows=1000, dtype={'gvkey': str})
fin_cols = ['size', 'leverage', 'bm', 'profitability', 'investment', 'rd', 'capex']
ret_cols = ['ret', 'ret_future_1q', 'ret_future_4q', 'momentum']
print('Financial variables:')
print(panel[fin_cols].describe().round(4))
print('\nReturn variables:')
print(panel[ret_cols].describe().round(4))

## Step 5: Final Deliverables

导出最终交付文件到 `outputs/tables/`。

脚本：`src/06_export_deliverables.py`

In [ ]:
# 检查主表
main = pd.read_csv(BASE / 'outputs' / 'tables' / 'firm_quarter_panel.csv',
                     nrows=10, dtype={'gvkey': str})
print(f'Main table columns ({len(main.columns)}):')
print(list(main.columns))
main.head()

In [ ]:
# 检查数据质量报告
qr = pd.read_csv(BASE / 'outputs' / 'tables' / 'data_quality_report.csv')
qr

In [ ]:
# 列出所有输出文件
import os
for root, dirs, files in os.walk(BASE / 'outputs'):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1e6
        print(f'  {os.path.relpath(fpath, BASE):50s} {size:8.1f} MB')